# 🧪 Scients: Scientific UEBA Anomaly Detection

Transformer (BERT MLM) vs Baselines: LSTM (DeepLog), Isolation Forest, One-Class SVM

## Quick Start
1. **Upload** `scients_package.zip` → Run All Cells
2. **(Optional)** Upload `BGL.log_structured.csv` for **real labels** (314K anomalies)
3. Without CSV → synthetic data generated automatically

### What's New (v3)
- **LSTM Neural Baseline (DeepLog-style)** — экспериментальное обоснование выбора Transformer
- **Ablation Study** — вклад Center Loss / Stochastic Scoring / Max-агрегации
- **Per-Category Breakdown** — detection rate по типам аномалий BGL (KERNDTLB, KERNSTOR, ...)
- **Explainability** — топ-N неожиданных событий аномальной сессии для аналитика
- **PR-AUC** — метрика для несбалансированных классов
- **Multi-seed Protocol** — mean±std по нескольким сидам (results_log.csv + aggregate_results.py)

### v2
- Real BGL Labels • IF/OC-SVM baselines • No Data Leakage (4-way split) • Full Reproducibility

In [ ]:
# Unzip the package
import os
if os.path.exists('scients_package.zip'):
    !unzip -o scients_package.zip
    print("Package unzipped successfully.")
else:
    print("Warning: scients_package.zip not found. Assuming files are already present.")

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU. Training might be slow.")

In [ ]:
# --- CONFIGURATION ---
SCIENTIFIC_MODE = False   # True = HPO enabled (slower but better results)
RUN_BASELINES = True      # True = compare against IF + OC-SVM
RUN_LSTM = True           # True = LSTM (DeepLog-style) neural baseline
EPOCHS = 10
BATCH_SIZE = 32
SEED = 42

In [ ]:
cmd = f"python main.py --epochs {EPOCHS} --batch_size {BATCH_SIZE} --seed {SEED}"
if not SCIENTIFIC_MODE:
    cmd += " --no_hpo"
if not RUN_BASELINES:
    cmd += " --no_baselines"
if not RUN_LSTM:
    cmd += " --no_lstm"

if os.path.exists("BGL.log_structured.csv"):
    print("✅ Using REAL BGL dataset with REAL anomaly labels")
else:
    print("⚠️ BGL CSV not found → using SYNTHETIC data generation")
    
print(f"Executing: {cmd}")
!{cmd}

## 📊 Results

In [ ]:
from IPython.display import Image, display
import pandas as pd

# Confusion Matrix + Score Distribution
if os.path.exists("performance.png"):
    print("=== Transformer Performance ===")
    display(Image(filename="performance.png"))

# Comparative Bar Chart (Transformer vs Baselines)
if os.path.exists("comparison.png"):
    print("\n=== Model Comparison ===")
    display(Image(filename="comparison.png"))

# Detection rate по категориям аномалий BGL (таблица для диплома)
if os.path.exists("per_category_results.csv"):
    print("\n=== Per-Category Anomaly Detection ===")
    display(pd.read_csv("per_category_results.csv"))

## 🧬 Ablation Study (глава 3 диплома)

Обучает две модели (с Center Loss и без) и сравнивает варианты скоринга:
3 прохода vs 1, max vs mean. Порог каждого варианта заново MCC-тюнится на Tune-Val.

In [ ]:
RUN_ABLATION = False  # True = запустить ablation (обучение ×2)

if RUN_ABLATION:
    !python run_ablation.py --epochs {EPOCHS} --seed {SEED}
    if os.path.exists("ablation_results.csv"):
        display(pd.read_csv("ablation_results.csv"))

## 🎲 Multi-seed протокол (mean ± std)

Несколько полных прогонов с разными сидами — оценка устойчивости результатов.
Долго на полном BGL: можно ускорить через --sample_rate 0.3.

In [ ]:
RUN_MULTISEED = False  # True = 3 полных прогона

if RUN_MULTISEED:
    for seed in [1, 2, 3]:
        # удаляем checkpoint, чтобы каждый сид обучался с нуля
        if os.path.exists("checkpoint.pth"):
            os.remove("checkpoint.pth")
        !python main.py --epochs {EPOCHS} --seed {seed} --tag multiseed --no_hpo --no_baselines --no_lstm
    !python aggregate_results.py --tag multiseed

## 📈 Training Logs (TensorBoard)
Uncomment the cell below to view training curves interactively.

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir runs